# E2a — Perovskite-specialized ChargeFlow (Colab, resumable)

**Goal:** chemistry-homogeneous proof-of-principle — `<1%` ε_MAE on a parent-disjoint perovskite split.

**Runtime:** Runtime → Change runtime type → **T4 GPU (free tier)**. Do NOT pay for A100/L4.

> Each perovskite has a different 3D grid size, so training uses `batch_size=1`
> with `accum_iter=16` for an effective batch of 16. The model is tiny
> (`unet_cond`, model_channels=4) — well under T4's 16 GB even at native resolution.

**Estimated cost:** $0 — free Colab T4 is sufficient for all 300 epochs.

## 0. Verify GPU

In [1]:
!nvidia-smi


Tue Jun 23 11:57:24 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 1. Mount Google Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/chargeflow'
DATA_DIR   = f'{DRIVE_ROOT}/data_perovskite'
OUTPUT_DIR = f'{DRIVE_ROOT}/output_perovskite'
LISTS_DIR  = f'{DRIVE_ROOT}/lists_perovskite'
for d in (DRIVE_ROOT, DATA_DIR, OUTPUT_DIR, LISTS_DIR):
    os.makedirs(d, exist_ok=True)
print('Drive ready:', DRIVE_ROOT)


Mounted at /content/drive
Drive ready: /content/drive/MyDrive/chargeflow


## 2. Clone repo

In [3]:
import os, shutil
REPO_DIR = '/content/chargeflow'

%cd /content

# Remove stale clone if present
if os.path.isdir(REPO_DIR):
    shutil.rmtree(REPO_DIR)
    print('Removed old clone.')

!git clone https://github.com/ngminhtri0394/chargeflow-electron-density-main.git {REPO_DIR}

# The GitHub repo may have a nested subfolder — find where scripts/ actually lives
actual_root = REPO_DIR
for entry in os.listdir(REPO_DIR):
    candidate = f'{REPO_DIR}/{entry}'
    if os.path.isdir(f'{candidate}/scripts'):
        actual_root = candidate
        print(f'Found scripts/ inside subfolder: {entry}')
        break
REPO_DIR = actual_root

%cd {REPO_DIR}
assert os.path.isdir(f'{REPO_DIR}/scripts'), f'scripts/ not found under {REPO_DIR}, contents: {os.listdir(REPO_DIR)}'
print('Done. Repo ready at', REPO_DIR)


/content
Cloning into '/content/chargeflow'...
remote: Enumerating objects: 275, done.
remote: Counting objects: 100% (275/275), done.
remote: Compressing objects: 100% (161/161), done.
remote: Total 275 (delta 149), reused 238 (delta 112), pack-reused 0 (from 0)
Receiving objects: 100% (275/275), 830.56 KiB | 17.30 MiB/s, done.
Resolving deltas: 100% (149/149), done.
Found scripts/ inside subfolder: chargeflow-electron-density-main
/content/chargeflow/chargeflow-electron-density-main
Done. Repo ready at /content/chargeflow/chargeflow-electron-density-main


In [7]:
# Install all deps: use pip install -e . first (picks up setup.py),
# then add packages not listed in requirements.txt that src/ actually imports.
!pip -q install -e .
!pip -q install flow_matching ase pymatgen torchmetrics torchvision pyyaml
import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())


  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.1/95.1 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 95.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.9/57.9 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.7/89.7 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 134.1 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.0/76.0 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 73.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 269.8/269.8 kB 28.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.1/121.1 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.6/85.6 kB 10.2 MB/s eta 0:00:00
     ━━

## 3. Write config (embedded — no upload needed)

The config lives directly in this cell so it works even on a fresh clone that doesn't have `config/train_perovskite_specialized.yaml`.

In [15]:
import yaml, os

cfg = {
    'model': {
        'architecture': 'unet_cond_xlarge',   # C=16; resolves near-core detail
        'discrete_flow_matching': False,
        'use_ema': True,
    },
    'data': {
        'train_data_list':      f'{LISTS_DIR}/list_d_perovskite_train',
        'train_label_list':     f'{LISTS_DIR}/list_l_perovskite_train',
        'train_data_gridsize':  f'{LISTS_DIR}/list_dgs_perovskite_train',
        'train_label_gridsize': f'{LISTS_DIR}/list_lgs_perovskite_train',
        'use_charge_dataset':   True,
        'data_augmentation':    True,
        'downsample_data':      1,
        'downsample_label':     1,
        'normalize_density':    False,
        'batch_size':           1,     # variable 3D grids
        'num_workers':          2,
        'pin_memory':           True,
    },
    'training': {
        'epochs':           2000,
        'start_epoch':      0,
        'optimizer':        'AdamW',
        'learning_rate':    1e-4,
        'optimizer_betas':  [0.9, 0.95],
        'decay_lr':         True,
        'accum_iter':       16,
        'class_drop_prob':  0.1,
        # *** KEY FIX: paper's formulation is SAD -> DFT flow, not noise -> residual. ***
        # start_sad=True makes the flow start from the SAD density (x_0=SAD, x_1=DFT).
        'start_sad':        True,
        'core_weight':      1.0,       # near-core-weighted loss (plan R1.3)
        'save_frequency':   10,
        'save_best':        True,
        'log_frequency':    10,
    },
    'evaluation': {'eval_frequency': 50, 'compute_fid': False, 'save_samples': True},
    'ode': {
        'method': 'dopri5', 'options': {'atol': 1e-5, 'rtol': 1e-5},
        'cfg_scale': 1.0, 'sampling_dtype': 'float32',
        'skewed_timesteps': True, 'edm_schedule': True,
    },
    'distributed': {'enabled': False, 'backend': 'nccl', 'init_method': 'env://'},
    'output': {
        'output_dir':   OUTPUT_DIR,
        'save_postfix': 'Perovskite_E2a_sad',   # new postfix -> fresh run
        'log_file':     'training_perovskite_sad.log',
    },
    'seed':      42,
    'device':    'cuda',
    'test_run':  False,
    'eval_only': False,
}

RUN_CFG = f'{OUTPUT_DIR}/run_config.yaml'
with open(RUN_CFG, 'w') as fh:
    yaml.safe_dump(cfg, fh, sort_keys=False)
print('Config written to', RUN_CFG)
print('start_sad:', cfg['training']['start_sad'], '| arch:', cfg['model']['architecture'],
      '| core_weight:', cfg['training']['core_weight'])


Config written to /content/drive/MyDrive/chargeflow/output_perovskite/run_config.yaml
start_sad: True | arch: unet_cond_xlarge | core_weight: 1.0


## 4. Download + extract perovskite data (ONCE — lives on Drive)

Skip on later sessions — data is already on Drive if the marker file exists.

In [ ]:
import os
ZENODO = 'https://zenodo.org/records/19211405/files'
files = [
    'perovskites_inputs_initial_density.tar.gz',
    'perovskites_targets_dft_density.tar.gz',
    'external_periodic_test_perovskites_split.csv',
    'paper_splits_all.csv',
]
marker = f'{DATA_DIR}/.extracted'
if not os.path.exists(marker):
    for f in files:
        local = f'/content/{f}'
        !wget -q -c "{ZENODO}/{f}" -O "{local}"
        if f.endswith('.tar.gz'):
            !tar -xzf "{local}" -C "{DATA_DIR}"
        else:
            !cp "{local}" "{DATA_DIR}/"
    open(marker, 'w').close()
    print('Extracted to', DATA_DIR)
else:
    print('Already extracted — skipping.')


## 5. Inspect extracted filenames

Run once to confirm folder/file structure before building splits.

In [ ]:
import glob, os
npys = glob.glob(f'{DATA_DIR}/**/*.npy', recursive=True)
print('total .npy:', len(npys))
print('--- first 10 ---')
for p in sorted(npys)[:10]:
    print(p)
# show top-level subdirs
print('\n--- top-level subdirs ---')
for d in sorted(os.listdir(DATA_DIR)):
    full = f'{DATA_DIR}/{d}'
    if os.path.isdir(full):
        n = len(os.listdir(full))
        print(f'  {d}/  ({n} entries)')


## 6. Build parent-disjoint train/test split + dataset file lists

Based on actual filenames from Colab output:
```
perovskites_rho_initial/VASP_mp-1055491_Rb1Ag1F3_remove_23_charge_-1/rho_22.npy
```
- Inputs folder: `perovskites_rho_initial/`
- Targets folder: `perovskites_rho_dft/` (or similar — confirmed by the inspect cell above)
- Pairing key: subfolder name `VASP_mp-XXXXX_..._charge_N`
- Parent id: `mp-XXXXX` (first MP id in the subfolder name)

The gridsize is recovered from the `.npy` array shape (written as a sidecar `.txt`).

In [ ]:
import re, glob, random, os
import numpy as np
random.seed(42)

# --- identify input and target root dirs ---
# 'perovskites_rho_initial' = SAD/neutral inputs ; 'perovskites_rho' = DFT targets
subdirs = [d for d in os.listdir(DATA_DIR)
           if os.path.isdir(f'{DATA_DIR}/{d}') and d.startswith('perovskites')]
print('perovskite subdirs:', subdirs)

INPUT_SUBDIR  = next((d for d in subdirs if 'initial' in d or 'input' in d), None)
TARGET_SUBDIR = next((d for d in subdirs if d != INPUT_SUBDIR), None)
assert INPUT_SUBDIR and TARGET_SUBDIR, f'Cannot resolve input/target in {subdirs}'
print('inputs :', INPUT_SUBDIR, '\ntargets:', TARGET_SUBDIR)

INPUT_ROOT  = f'{DATA_DIR}/{INPUT_SUBDIR}'
TARGET_ROOT = f'{DATA_DIR}/{TARGET_SUBDIR}'

# --- pair input/target by matching subfolder name ---
def find_npy(root_dir):
    result = {}
    for subdir in os.listdir(root_dir):
        p = f'{root_dir}/{subdir}'
        if os.path.isdir(p):
            npys = glob.glob(f'{p}/*.npy')
            if npys:
                result[subdir] = npys[0]
    return result

inp_map = find_npy(INPUT_ROOT)
tgt_map = find_npy(TARGET_ROOT)
common  = sorted(set(inp_map) & set(tgt_map))
pairs   = [(inp_map[k], tgt_map[k]) for k in common]
print(f'paired: {len(pairs)}')

def parent_id_of(subdir_name):
    m = re.search(r'(mp-\d+)', subdir_name)
    return m.group(1) if m else subdir_name

# --- gridsize sidecar from the REAL .npy shape (arrays are already 3D) ---
# The dataset does np.load(npy).reshape(1, *gridsize), so gridsize MUST equal
# the true array shape. We read it directly — no guessing.
def gridsize_path_for(npy_path):
    txt = npy_path.replace('.npy', '.realshape.txt')
    arr = np.load(npy_path, mmap_mode='r')
    assert arr.ndim == 3, f'Expected 3D array, got {arr.ndim}D for {npy_path}'
    with open(txt, 'w') as fh:
        fh.write(' '.join(map(str, arr.shape)))
    return txt

# --- parent-disjoint 85/15 split ---
parents  = sorted({parent_id_of(k) for k in common})
random.shuffle(parents)
n_test   = max(1, int(0.15 * len(parents)))
test_set = set(parents[:n_test])
train_pairs = [(i, t) for k, (i, t) in zip(common, pairs) if parent_id_of(k) not in test_set]
test_pairs  = [(i, t) for k, (i, t) in zip(common, pairs) if parent_id_of(k) in test_set]
print(f'parents: {len(parents)}  train: {len(train_pairs)}  test: {len(test_pairs)}')

def write_lists(split_pairs, tag):
    data_paths  = [p[0] for p in split_pairs]
    label_paths = [p[1] for p in split_pairs]
    data_gs     = [gridsize_path_for(p) for p in data_paths]
    label_gs    = [gridsize_path_for(p) for p in label_paths]
    for name, items in [
        (f'list_d_{tag}',   data_paths),
        (f'list_l_{tag}',   label_paths),
        (f'list_dgs_{tag}', data_gs),
        (f'list_lgs_{tag}', label_gs),
    ]:
        with open(f'{LISTS_DIR}/{name}', 'w') as fh:
            fh.write('\n'.join(items) + '\n')
    print(f'wrote {tag} lists ({len(split_pairs)} samples)')

write_lists(train_pairs, 'perovskite_train')
write_lists(test_pairs,  'perovskite_test')

# sanity check: confirm a written gridsize matches the npy shape
chk_npy = data_paths_check = open(f'{LISTS_DIR}/list_d_perovskite_train').readline().strip()
chk_gs  = open(f'{LISTS_DIR}/list_dgs_perovskite_train').readline().strip()
print('\nSanity:', np.load(chk_npy, mmap_mode='r').shape, '==', open(chk_gs).read())


In [ ]:
# DIAGNOSTIC: check the actual shape of the .npy density files and whether
# the original grid_sizes_*.dat files were shipped with the data.
import glob, os, numpy as np

sample_dir = glob.glob(f'{DATA_DIR}/perovskites_rho_initial/*/')[0]
print('Sample structure folder:', sample_dir)
print('Files in it:', os.listdir(sample_dir))
print()

npy = glob.glob(f'{sample_dir}/*.npy')[0]
arr = np.load(npy)
print(f'{os.path.basename(npy)}: shape={arr.shape}, ndim={arr.ndim}, size={arr.size}')
if arr.ndim == 1:
    cube = round(arr.size ** (1/3))
    print(f'  flat array. cube-root = {cube} (cube={cube**3==arr.size})')
    print('  -> need the real grid_sizes_*.dat to reshape correctly!')

# Check a few more structures to see if shapes/sizes differ across the family
print('\nSizes across 8 structures:')
for d in sorted(glob.glob(f'{DATA_DIR}/perovskites_rho_initial/*/'))[:8]:
    p = glob.glob(f'{d}/*.npy')[0]
    a = np.load(p, mmap_mode='r')
    print(f'  {os.path.basename(d.rstrip("/"))[:40]:40s} shape={a.shape} size={a.size}')


## 7. Find latest checkpoint (auto-resume)

Run this at the start of every session — it picks up where the last session left off.

In [5]:
# Pull latest code fixes (REPO_DIR is the nested path resolved in cell 2)
!git -C {REPO_DIR} pull

# Find latest checkpoint for the CURRENT run's postfix, for auto-resume.
import glob, os
POSTFIX = 'Perovskite_E2a_sad'   # must match cfg['output']['save_postfix'] in cell 3
rolling = f'{OUTPUT_DIR}/checkpoint-charged-residual-{POSTFIX}.pth'
RESUME  = rolling if os.path.exists(rolling) else None
if RESUME is None:
    ck = glob.glob(f'{OUTPUT_DIR}/checkpoint-*-{POSTFIX}.pth')
    if ck:
        RESUME = max(ck, key=os.path.getmtime)
print('Resume from:', RESUME or '(none — fresh start)')


Already up to date.
Resume from: /content/drive/MyDrive/chargeflow/output_perovskite/checkpoint-charged-residual-Perovskite_E2a_sad.pth


In [ ]:
!git -C /content/chargeflow pull


## 8. Train (resumable)

Runs until `epochs` or until you stop the cell. Each disconnect is safe — the last Drive checkpoint is intact.

**After your first epoch completes**, note the epoch time printed in the log and adjust `save_frequency` in the config dict above (cell 3) so checkpoints land every ~15 min, then re-run cell 3 + this cell.

In [ ]:
resume_arg = f'--resume "{RESUME}"' if RESUME else ''
!cd {REPO_DIR} && python scripts/train.py --config "{RUN_CFG}" {resume_arg}


## 9. Final evaluation — the editor-gate ε_MAE number

Run after training converges. Uses the parent-disjoint test split. Target: **< 0.01 (1%)**.

In [8]:
# Editor-gate evaluation (start_sad=True / SAD->DFT flow, the paper's formulation).
# ODE starts from the SAD input (x_0 = SAD), integrates to the DFT density.
# Output IS the DFT prediction directly (NO + SAD). Charge index -> label,
# concat_conditioning = SAD (Option B: model gets SAD both as flow start AND concat).
# ε_MAE = sum(|pred - target|) / sum(target), averaged over structures.
import sys, os
for m in [k for k in list(sys.modules) if k == 'src' or k.startswith('src.')]:
    del sys.modules[m]
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

import torch, numpy as np
from src.data.dataset import RhoDatasetCharge
from src.models.model_configs import instantiate_model
from flow_matching.solver.ode_solver import ODESolver
from flow_matching.utils import ModelWrapper
from src.training.edm_time_discretization import get_time_discretization

class TrainArgs:  # stub so torch.load can unpickle the checkpoint's args
    def __init__(self, *a, **k): pass

device = torch.device('cuda')
NFE = 100
START_SAD = True                  # paper's SAD->DFT formulation
ARCH    = 'unet_cond_xlarge'      # match what you trained
POSTFIX = 'Perovskite_E2a_sad'    # match cfg save_postfix for the start_sad run

ckpt_path = f'{OUTPUT_DIR}/best_model-{POSTFIX}.pth'
if not os.path.exists(ckpt_path):
    ckpt_path = f'{OUTPUT_DIR}/checkpoint-charged-residual-{POSTFIX}.pth'
model = instantiate_model(architechture=ARCH, is_discrete=False, use_ema=True)
ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
state = ckpt['model'] if 'model' in ckpt else ckpt
if list(state.keys())[0].startswith('module.'):
    state = {k.replace('module.', ''): v for k, v in state.items()}
model.load_state_dict(state, strict=True); model.to(device).eval()
print(f"Checkpoint: {os.path.basename(ckpt_path)} (epoch {ckpt.get('epoch','?')})  arch={ARCH}")

class ChargeCFGModel(ModelWrapper):
    def forward(self, x, t, charge, concat):
        t = torch.zeros(x.shape[0], device=x.device) + t
        with torch.autocast('cuda'), torch.no_grad():
            return self.model(x, t, extra={"label": charge,
                                           "concat_conditioning": concat}).to(torch.float32)
wrapped = ChargeCFGModel(model); wrapped.eval()
solver = ODESolver(velocity_model=wrapped)

test_ds = RhoDatasetCharge(
    data_list_path     = f'{LISTS_DIR}/list_d_perovskite_test',
    label_list_path    = f'{LISTS_DIR}/list_l_perovskite_test',
    data_gridsize_path = f'{LISTS_DIR}/list_dgs_perovskite_test',
    label_gridsize_path= f'{LISTS_DIR}/list_lgs_perovskite_test',
    data_augmentation  = False,
)
loader = torch.utils.data.DataLoader(test_ds, batch_size=1, shuffle=False)
print(f'Test structures: {len(test_ds)}')

eps_list = []
time_grid = get_time_discretization(nfes=NFE).to(device)
for i, (sad, target, charge) in enumerate(loader):
    sad = sad.to(device); target = target.to(device).float(); charge = charge.to(device)
    x_0 = sad if START_SAD else torch.randn_like(sad)*torch.sqrt(torch.tensor(1e-05, device=device))
    with torch.no_grad():
        out = solver.sample(time_grid=time_grid, x_init=x_0, method='dopri5',
                            atol=1e-5, rtol=1e-5, step_size=None,
                            charge=charge, concat=sad).float()
    pred = out if START_SAD else (sad + out)   # start_sad: output IS the DFT density
    eps = (pred - target).abs().sum() / target.sum()
    eps_list.append(eps.item())
    print(f'  [{i+1:2d}/{len(test_ds)}] ε_MAE = {eps.item()*100:.3f}%')

import statistics
mean_eps = statistics.mean(eps_list)
print('\n' + '='*60)
print(f'  Mean ε_MAE: {mean_eps*100:.3f}%   median: {statistics.median(eps_list)*100:.3f}%')
print(f'  frac < 1%: {sum(e<0.01 for e in eps_list)/len(eps_list)*100:.0f}%')
print(f'  Editor gate (<1%): {"PASS" if mean_eps < 0.01 else "not yet"}')
print('='*60)


Checkpoint: best_model-Perovskite_E2a_sad.pth (epoch 1874)  arch=unet_cond_xlarge
Test structures: 24
  [ 1/24] ε_MAE = 8.485%
  [ 2/24] ε_MAE = 8.710%
  [ 3/24] ε_MAE = 8.097%
  [ 4/24] ε_MAE = 8.128%
  [ 5/24] ε_MAE = 5.572%
  [ 6/24] ε_MAE = 5.368%
  [ 7/24] ε_MAE = 5.209%
  [ 8/24] ε_MAE = 5.229%
  [ 9/24] ε_MAE = 5.417%
  [10/24] ε_MAE = 5.256%
  [11/24] ε_MAE = 5.171%
  [12/24] ε_MAE = 5.024%
  [13/24] ε_MAE = 4.848%
  [14/24] ε_MAE = 4.985%
  [15/24] ε_MAE = 4.736%
  [16/24] ε_MAE = 4.584%
  [17/24] ε_MAE = 4.465%
  [18/24] ε_MAE = 4.371%
  [19/24] ε_MAE = 4.050%
  [20/24] ε_MAE = 4.241%
  [21/24] ε_MAE = 6.125%
  [22/24] ε_MAE = 6.517%
  [23/24] ε_MAE = 5.906%
  [24/24] ε_MAE = 6.165%

  Mean ε_MAE: 5.694%   median: 5.243%
  frac < 1%: 0%
  Editor gate (<1%): not yet


In [9]:
# OVERFIT CHECK: eval ε_MAE on a few TRAINING structures.
# Uses the SAME SAD->DFT flow as cell 23 (start_sad=True): x_0 = SAD, output IS the
# DFT density (no + SAD). Earlier version started from noise and added SAD back — that
# was the OLD noise->residual flow and gave a bogus ~25% train ε_MAE. Now apples-to-apples.
# train≈test -> not overfitting; loss objective doesn't yield low ε_MAE (need bigger model / near-core loss).
# train<<test -> overfitting (need more data / regularization).
import torch, statistics
from src.data.dataset import RhoDatasetCharge

train_ds = RhoDatasetCharge(
    data_list_path     = f'{LISTS_DIR}/list_d_perovskite_train',
    label_list_path    = f'{LISTS_DIR}/list_l_perovskite_train',
    data_gridsize_path = f'{LISTS_DIR}/list_dgs_perovskite_train',
    label_gridsize_path= f'{LISTS_DIR}/list_lgs_perovskite_train',
    data_augmentation  = False,
)
train_loader = torch.utils.data.DataLoader(train_ds, batch_size=1, shuffle=False)

tg = get_time_discretization(nfes=NFE).to(device)
train_eps = []
for i, (sad, target, charge) in enumerate(train_loader):
    sad = sad.to(device); target = target.to(device).float(); charge = charge.to(device)
    x_0 = sad if START_SAD else torch.randn_like(sad) * torch.sqrt(torch.tensor(1e-05, device=device))
    with torch.no_grad():
        out = solver.sample(time_grid=tg, x_init=x_0, method='dopri5',
                            atol=1e-5, rtol=1e-5, step_size=None,
                            charge=charge, concat=sad).float()
    pred = out if START_SAD else (sad + out)   # start_sad: output IS the DFT density
    train_eps.append(((pred - target).abs().sum() / target.sum()).item())
    if i >= 9:   # first 10 training structures is enough
        break

print(f'TRAIN ε_MAE (first 10): mean {statistics.mean(train_eps)*100:.2f}%')
print(f'TEST  ε_MAE (all 24):   mean {mean_eps*100:.2f}%')
print()
if statistics.mean(train_eps) < 0.5 * mean_eps:
    print('=> OVERFITTING: train much better than test. Need more data / regularization.')
else:
    print('=> NOT overfitting: train ~= test. Loss objective does not yield low ε_MAE.')
    print('   Levers: near-core-weighted loss, bigger model (unet_cond_xlarge).')


TRAIN ε_MAE (first 10): mean 3.69%
TEST  ε_MAE (all 24):   mean 5.69%

=> NOT overfitting: train ~= test. Loss objective does not yield low ε_MAE.
   Levers: near-core-weighted loss, bigger model (unet_cond_xlarge).


In [10]:
# Was the loss still descending at epoch 285, or had it plateaued?
# Reads the training log to decide if "more epochs" is worthwhile.
import json, os
log_path = f'{OUTPUT_DIR}/log.txt'
if os.path.exists(log_path):
    rows = [json.loads(l) for l in open(log_path) if l.strip()]
    losses = [(r.get('epoch'), r.get('train_loss')) for r in rows if 'train_loss' in r]
    print(f'{len(losses)} logged epochs')
    for e, l in losses[:5]:   print(f'  epoch {e:3d}: {l:.5f}')
    print('  ...')
    for e, l in losses[-10:]: print(f'  epoch {e:3d}: {l:.5f}')
else:
    print('No log.txt found at', log_path)


2826 logged epochs
  epoch   0: 0.05755
  epoch   1: 0.05681
  epoch   2: 0.05569
  epoch   3: 0.05412
  epoch   4: 0.05209
  ...
  epoch 1900: 0.00567
  epoch 1901: 0.00570
  epoch 1902: 0.00565
  epoch 1903: 0.00572
  epoch 1904: 0.00568
  epoch 1905: 0.00569
  epoch 1906: 0.00569
  epoch 1907: 0.00569
  epoch 1908: 0.00568
  epoch 1909: 0.00567


In [11]:
# DIAGNOSTIC: disambiguate "model bad" vs "eval wrong".
# Uses the SAME SAD->DFT flow as cell 23 (start_sad=True): x_0 = SAD, output IS the
# predicted DFT density. The model "correction" we compare is (pred - sad).
# 1) SAD baseline: eps_MAE of the raw input vs target (no model). The model must beat this.
# 2) correction magnitude: is the model moving SAD toward DFT, or barely moving it?
# 3) compare to the TRUE correction (target - sad) magnitude.
import torch, statistics

base_eps, corr_pred_mag, corr_true_mag = [], [], []
time_grid_d = get_time_discretization(nfes=NFE).to(device)
for i, (sad, target, charge) in enumerate(loader):
    sad = sad.to(device); target = target.to(device).float(); charge = charge.to(device)
    # baseline: just SAD
    base_eps.append(((sad - target).abs().sum() / target.sum()).item())
    # what the model predicted (SAD->DFT flow: output IS the DFT density)
    x_0 = sad if START_SAD else torch.randn_like(sad) * torch.sqrt(torch.tensor(1e-05, device=device))
    with torch.no_grad():
        out = solver.sample(time_grid=time_grid_d, x_init=x_0, method='dopri5',
                            atol=1e-5, rtol=1e-5, step_size=None,
                            charge=charge, concat=sad).float()
    pred = out if START_SAD else (sad + out)
    corr_pred_mag.append((pred - sad).abs().mean().item())   # how far the model moved SAD
    corr_true_mag.append((target - sad).abs().mean().item()) # how far it SHOULD move
    if i >= 5:
        break

print(f'SAD-only baseline ε_MAE (first 6): mean {statistics.mean(base_eps)*100:.2f}%')
print(f'Model correction |mean abs| (pred-sad, first 6): {statistics.mean(corr_pred_mag):.3e}')
print(f'TRUE correction |mean abs| (target-sad): {statistics.mean(corr_true_mag):.3e}')
print()
print('Interpretation:')
print(' - If SAD baseline ~= model eps: model is not improving on the input.')
print(' - If pred correction << true correction: model under-moves SAD (weak/under-trained).')
print(' - If pred correction ~ true correction but eps high: it moves the wrong places.')


SAD-only baseline ε_MAE (first 6): mean 19.37%
Model correction |mean abs| (pred-sad, first 6): 1.128e-01
TRUE correction |mean abs| (target-sad): 1.274e-01

Interpretation:
 - If SAD baseline ~= model eps: model is not improving on the input.
 - If pred correction << true correction: model under-moves SAD (weak/under-trained).
 - If pred correction ~ true correction but eps high: it moves the wrong places.


In [12]:
# ROOT-CAUSE CHECK: are SAD input and DFT target on the same scale?
# And what is the BEST POSSIBLE ε_MAE if the model predicted the TRUE residual exactly?
import torch, numpy as np, statistics

ideal_eps, sad_sum, dft_sum = [], [], []
for i, (sad, target, charge) in enumerate(loader):
    sad = sad.float(); target = target.float()
    sad_sum.append(sad.sum().item()); dft_sum.append(target.sum().item())
    # if model predicted the EXACT residual, pred = sad + (target - sad) = target -> eps=0.
    # So a nonzero "ideal" here would reveal a metric/scale bug. Check identity:
    ideal_pred = sad + (target - sad)
    ideal_eps.append(((ideal_pred - target).abs().sum() / target.sum()).item())
    if i >= 7: break

print('SAD   total (electrons):', [f'{s:.1f}' for s in sad_sum])
print('DFT   total (electrons):', [f'{s:.1f}' for s in dft_sum])
print('ratio DFT/SAD          :', [f'{d/s:.3f}' for s,d in zip(sad_sum,dft_sum)])
print(f'\n"ideal" ε_MAE (pred=exact residual): {statistics.mean(ideal_eps)*100:.4f}%  (must be ~0)')
print('\nIf DFT/SAD ratio is far from 1.0 -> different normalization/electron count.')
print('If totals look like raw electron counts (tens-hundreds) -> scales comparable.')


SAD   total (electrons): ['551201.3', '549639.9', '554324.3', '555885.8', '349059.0', '347741.8', '351693.4', '353010.6']
DFT   total (electrons): ['551201.4', '549639.9', '554324.3', '555885.8', '349058.9', '347741.8', '351693.4', '353010.6']
ratio DFT/SAD          : ['1.000', '1.000', '1.000', '1.000', '1.000', '1.000', '1.000', '1.000']

"ideal" ε_MAE (pred=exact residual): 0.0000%  (must be ~0)

If DFT/SAD ratio is far from 1.0 -> different normalization/electron count.
If totals look like raw electron counts (tens-hundreds) -> scales comparable.


## 10. Strategy experiments — closing the gap to <1% ε_MAE

Corrected diagnosis (after the cell-24/26 flow fix):
- **TRAIN 3.7% vs TEST 5.7%** → mild overfitting on 135 samples, NOT a broken objective.
- **Model correction 0.113 vs true 0.127** → model moves SAD ~89% of the way; the missing
  ~10% is fine near-core/bonding structure, where ε_MAE mass concentrates.

Three code-supported levers **never exercised** in the runs so far (all confirmed in
`src/training/train_loop.py`):
1. **`core_weight`** — density-weighted L2 (`w = 1 + core_weight·ρ/mean(ρ)`). Ran at `1.0` (mild,
   ~2× near core). Try 5–10.
2. **`loss_type='hybrid'` + `alpha`** — adds `alpha · NormMAE(one-step density estimate, target)`.
   `NormMAE = sum|pred-target|/sum(target)` = **exactly the ε_MAE gate metric.** Never enabled.
3. **`normalize_density` (norm_rho)** — log-compresses the huge near-core dynamic range. Never enabled.

Cells below: **(0a)** NFE sweep to bound solver error, **(0b)** spatial error map to confirm the
near-core hypothesis, **(1)** a config helper to launch short ablation retrains from the current checkpoint.

In [13]:
# EXPERIMENT 0a — NFE sweep: how much of the 5.7% is ODE-solver error vs model error?
# Re-runs the editor-gate eval at several NFE counts on a subset of the test set.
# If ε_MAE drops materially as NFE rises, part of the error is integration error (free win
# at inference). If flat, the solver is NOT the bottleneck and the 5.7% is genuine model error.
# Reuses: model/solver/wrapped (cell 23), loader, START_SAD, device, get_time_discretization.
import torch, statistics

NFE_GRID = [20, 50, 100, 250]
N_EVAL   = 8   # subset for speed; bump to len(test_ds) for the final number

results = {}
for nfe in NFE_GRID:
    tg = get_time_discretization(nfes=nfe).to(device)
    eps = []
    for i, (sad, target, charge) in enumerate(loader):
        if i >= N_EVAL:
            break
        sad = sad.to(device); target = target.to(device).float(); charge = charge.to(device)
        x_0 = sad if START_SAD else torch.randn_like(sad) * torch.sqrt(torch.tensor(1e-05, device=device))
        with torch.no_grad():
            out = solver.sample(time_grid=tg, x_init=x_0, method='dopri5',
                                atol=1e-5, rtol=1e-5, step_size=None,
                                charge=charge, concat=sad).float()
        pred = out if START_SAD else (sad + out)
        eps.append(((pred - target).abs().sum() / target.sum()).item())
    results[nfe] = statistics.mean(eps)
    print(f'  NFE={nfe:4d}: mean ε_MAE (first {N_EVAL}) = {results[nfe]*100:.3f}%')

spread = (max(results.values()) - min(results.values())) * 100
print()
print(f'ε_MAE spread across NFE: {spread:.3f} percentage points')
if spread < 0.2:
    print('=> Solver is NOT the bottleneck (flat in NFE). The error is genuine model error.')
else:
    print('=> Solver error is material. Use the best NFE for eval; also try tighter atol/rtol.')


  NFE=  20: mean ε_MAE (first 8) = 6.850%
  NFE=  50: mean ε_MAE (first 8) = 6.850%
  NFE= 100: mean ε_MAE (first 8) = 6.850%
  NFE= 250: mean ε_MAE (first 8) = 6.850%

ε_MAE spread across NFE: 0.000 percentage points
=> Solver is NOT the bottleneck (flat in NFE). The error is genuine model error.


In [14]:
# EXPERIMENT 0b — spatial error map: is ε_MAE dominated by high-density (near-core) voxels?
# Bins voxels by TARGET density percentile and reports each bin's share of total |pred-target|.
# If the top density bins carry most of the error -> near-core weighting (core_weight / norm_rho)
# is the right lever. If error is spread evenly -> weighting won't help; need capacity/data.
# Reuses model/solver/loader/START_SAD/device from cell 23.
import torch, numpy as np

NFE = 100
tg = get_time_discretization(nfes=NFE).to(device)
# density percentile edges (by target ρ): bulk -> near-core
edges_q = [0, 50, 90, 99, 99.9, 100]

err_by_bin = np.zeros(len(edges_q) - 1)
tot_by_bin = np.zeros(len(edges_q) - 1)
abs_err_total = 0.0
N_EVAL = 6
for i, (sad, target, charge) in enumerate(loader):
    if i >= N_EVAL:
        break
    sad = sad.to(device); target = target.to(device).float(); charge = charge.to(device)
    x_0 = sad if START_SAD else torch.randn_like(sad) * torch.sqrt(torch.tensor(1e-05, device=device))
    with torch.no_grad():
        out = solver.sample(time_grid=tg, x_init=x_0, method='dopri5',
                            atol=1e-5, rtol=1e-5, step_size=None,
                            charge=charge, concat=sad).float()
    pred = out if START_SAD else (sad + out)
    t = target.flatten().cpu().numpy()
    e = (pred - target).abs().flatten().cpu().numpy()
    abs_err_total += e.sum()
    qs = np.percentile(t, edges_q)
    qs[-1] = np.inf  # include the max
    for b in range(len(edges_q) - 1):
        mask = (t >= qs[b]) & (t < qs[b + 1])
        err_by_bin[b] += e[mask].sum()
        tot_by_bin[b] += t[mask].sum()

print(f'Spatial error decomposition (first {N_EVAL} test structures), by target-density percentile:')
print(f'{"density bin":>16} | {"% of total |err|":>16} | {"% of total ρ":>14}')
print('-' * 54)
labels = [f'{edges_q[b]:.4g}-{edges_q[b+1]:.4g}%' for b in range(len(edges_q) - 1)]
for b, lab in enumerate(labels):
    pct_err = err_by_bin[b] / err_by_bin.sum() * 100
    pct_rho = tot_by_bin[b] / tot_by_bin.sum() * 100
    print(f'{lab:>16} | {pct_err:>15.1f}% | {pct_rho:>13.1f}%')

top_share = (err_by_bin[-1] + err_by_bin[-2]) / err_by_bin.sum() * 100
print()
print(f'Top 1% densest voxels carry {top_share:.1f}% of the absolute error.')
if top_share > 50:
    print('=> Error IS near-core concentrated. core_weight↑ / norm_rho are the right levers.')
else:
    print('=> Error is spread across density. Weighting alone wont fix it; need capacity/data.')


Spatial error decomposition (first 6 test structures), by target-density percentile:
     density bin | % of total |err| |   % of total ρ
------------------------------------------------------
           0-50% |            17.2% |           9.5%
          50-90% |            55.4% |          34.1%
          90-99% |            19.8% |          37.9%
        99-99.9% |             6.0% |          15.3%
       99.9-100% |             1.6% |           3.1%

Top 1% densest voxels carry 7.6% of the absolute error.
=> Error is spread across density. Weighting alone wont fix it; need capacity/data.


In [16]:
# EXPERIMENT 1 — ablation launcher: write a config for a lever combo and train a SHORT run.
# Each experiment gets its OWN save_postfix -> its own checkpoint, so the epoch-1874 baseline
# (Perovskite_E2a_sad) is never touched. Warm-starts from the baseline checkpoint so each
# ablation only needs ~50-100 extra epochs to show whether the lever moves test ε_MAE.
#
# Config knobs exposed (all read by scripts/train.py -> TrainArgs, confirmed in src/training/train_loop.py):
#   core_weight : density-weighted L2.  loss_type/'alpha' : 'l2' | 'hybrid' | 'normmae'.
#   normalize_density : log-compress ρ (norm_rho).
import yaml, copy, os

# baseline config from cell 9 (re-run cell 9 first so `cfg` exists)
BASE = copy.deepcopy(cfg)
BASELINE_CKPT = f'{OUTPUT_DIR}/best_model-Perovskite_E2a_sad.pth'  # warm-start source

def make_experiment(name, *, core_weight=None, loss_type=None, alpha=None,
                    normalize_density=None, extra_epochs=80):
    """Write a run config for one ablation. Returns (config_path, postfix, resume_ckpt)."""
    c = copy.deepcopy(BASE)
    postfix = f'Perovskite_E2a_{name}'
    if core_weight       is not None: c['training']['core_weight'] = core_weight
    if loss_type         is not None: c['training']['loss_type']   = loss_type
    if alpha             is not None: c['training']['alpha']       = alpha
    if normalize_density is not None: c['data']['normalize_density'] = normalize_density
    # train a short increment past the baseline's epoch (1874) so we see the effect fast
    c['training']['epochs'] = 1874 + extra_epochs
    c['output']['save_postfix'] = postfix
    c['output']['log_file']     = f'training_{name}.log'
    path = f'{OUTPUT_DIR}/run_config_{name}.yaml'
    with open(path, 'w') as fh:
        yaml.safe_dump(c, fh, sort_keys=False)
    resume = BASELINE_CKPT if os.path.exists(BASELINE_CKPT) else None
    print(f'[{name}] -> {path}')
    print(f'   core_weight={c["training"].get("core_weight")}  '
          f'loss_type={c["training"].get("loss_type","l2")}  '
          f'alpha={c["training"].get("alpha")}  '
          f'norm_rho={c["data"]["normalize_density"]}  epochs->{c["training"]["epochs"]}')
    print(f'   warm-start: {resume or "(none)"}')
    return path, postfix, resume

# --- define the ablation grid (run the ones you want, one at a time) ---
EXPERIMENTS = {
    # 1) sharpen the existing near-core weighting
    'cw5'     : dict(core_weight=5.0),
    'cw10'    : dict(core_weight=10.0),
    # 2) optimize the actual metric directly (NormMAE == ε_MAE) alongside L2
    'hybrid'  : dict(loss_type='hybrid', alpha=2.0, core_weight=1.0),
    # 3) log-compress the dynamic range so near-core isn't drowned out
    'normrho' : dict(normalize_density=True, core_weight=1.0),
    # 4) combined best-guess
    'combo'   : dict(loss_type='hybrid', alpha=2.0, core_weight=5.0, normalize_density=True),
}

print('Defined experiments:', list(EXPERIMENTS.keys()))
print('NOTE: warm-starting from best_model changes the loss surface; if resume asserts on epoch,')
print('      set extra_epochs higher or start fresh (drop the resume arg in the train cell below).')


Defined experiments: ['cw5', 'cw10', 'hybrid', 'normrho', 'combo']
NOTE: warm-starting from best_model changes the loss surface; if resume asserts on epoch,
      set extra_epochs higher or start fresh (drop the resume arg in the train cell below).


In [ ]:
# EXPERIMENT 1 — RUN one ablation. Pick a name from EXPERIMENTS, then train.
# Warm-starts from the baseline best_model; saves under its own postfix on Drive (resumable).
EXP_NAME = 'hybrid'   # <- change to 'cw5', 'cw10', 'normrho', 'combo', ...

exp_cfg_path, EXP_POSTFIX, exp_resume = make_experiment(EXP_NAME, **EXPERIMENTS[EXP_NAME])

# If an ablation checkpoint already exists (you re-ran this session), resume from IT instead
# of the baseline so progress isn't lost.
exp_ckpt = f'{OUTPUT_DIR}/checkpoint-charged-residual-{EXP_POSTFIX}.pth'
resume_from = exp_ckpt if os.path.exists(exp_ckpt) else exp_resume
resume_arg  = f'--resume "{resume_from}"' if resume_from else ''
print('Training', EXP_NAME, '| resume from:', resume_from or '(fresh)')

!cd {REPO_DIR} && python scripts/train.py --config "{exp_cfg_path}" {resume_arg}


[hybrid] -> /content/drive/MyDrive/chargeflow/output_perovskite/run_config_hybrid.yaml
   core_weight=1.0  loss_type=hybrid  alpha=2.0  norm_rho=False  epochs->1954
   warm-start: /content/drive/MyDrive/chargeflow/output_perovskite/best_model-Perovskite_E2a_sad.pth
Training hybrid | resume from: /content/drive/MyDrive/chargeflow/output_perovskite/best_model-Perovskite_E2a_sad.pth
2026-06-23 12:27:19 [INFO    ] train: ================================================================================
2026-06-23 12:27:19 [INFO    ] train: Starting Training
2026-06-23 12:27:19 [INFO    ] train: ================================================================================
2026-06-23 12:27:19 [INFO    ] train: Output directory: /content/drive/MyDrive/chargeflow/output_perovskite
2026-06-23 12:27:21 [INFO    ] train: Saved configuration to /content/drive/MyDrive/chargeflow/output_perovskite/config.yaml
Not using distributed mode
2026-06-23 12:27:21 [INFO    ] train: Device: cuda
2026-06-23 1

In [ ]:
# EXPERIMENT 1 — EVAL an ablation checkpoint and compare to the 5.7% baseline.
# Re-instantiates the model and loads the experiment's checkpoint. Handles norm_rho:
# if the run used normalize_density=True, the flow lives in log-space, so x_0 = scaled SAD
# and the output is un-scaled back to real density before computing ε_MAE.
# Scaling matches src/training/train_loop.py:  s(ρ)=sign(ρ)·log(1+1.25|ρ|)/4.2 ;  inverse below.
import torch, statistics, os

EVAL_NAME    = EXP_NAME                      # which experiment to score
EVAL_POSTFIX = f'Perovskite_E2a_{EVAL_NAME}'
EVAL_NORMRHO = bool(EXPERIMENTS[EVAL_NAME].get('normalize_density', False))
NFE = 100

def scale_rho(r):    # forward log-compression (train_loop.py:131)
    return r.sign() * torch.log1p(1.25 * r.abs()) / 4.2
def unscale_rho(s):  # inverse: ρ = sign(s)·(exp(4.2|s|)-1)/1.25
    return s.sign() * (torch.expm1(4.2 * s.abs())) / 1.25

ck = f'{OUTPUT_DIR}/best_model-{EVAL_POSTFIX}.pth'
if not os.path.exists(ck):
    ck = f'{OUTPUT_DIR}/checkpoint-charged-residual-{EVAL_POSTFIX}.pth'
assert os.path.exists(ck), f'No checkpoint for {EVAL_NAME} yet — run the train cell first.'

em = instantiate_model(architechture=ARCH, is_discrete=False, use_ema=True)
cp = torch.load(ck, map_location=device, weights_only=False)
st = cp['model'] if 'model' in cp else cp
if list(st.keys())[0].startswith('module.'):
    st = {k.replace('module.', ''): v for k, v in st.items()}
em.load_state_dict(st, strict=True); em.to(device).eval()
ew = ChargeCFGModel(em); ew.eval()
esolver = ODESolver(velocity_model=ew)
print(f'[{EVAL_NAME}] checkpoint {os.path.basename(ck)} (epoch {cp.get("epoch","?")})  norm_rho={EVAL_NORMRHO}')

tg = get_time_discretization(nfes=NFE).to(device)
eps = []
for i, (sad, target, charge) in enumerate(loader):
    sad = sad.to(device); target = target.to(device).float(); charge = charge.to(device)
    x_0 = scale_rho(sad) if EVAL_NORMRHO else sad
    cc  = x_0                                   # concat conditioning matches flow space
    with torch.no_grad():
        out = esolver.sample(time_grid=tg, x_init=x_0, method='dopri5',
                             atol=1e-5, rtol=1e-5, step_size=None,
                             charge=charge, concat=cc).float()
    pred = unscale_rho(out) if EVAL_NORMRHO else out
    eps.append(((pred - target).abs().sum() / target.sum()).item())

m = statistics.mean(eps)
print(f'  [{EVAL_NAME}] mean ε_MAE: {m*100:.3f}%   median: {statistics.median(eps)*100:.3f}%   '
      f'frac<1%: {sum(e<0.01 for e in eps)/len(eps)*100:.0f}%')
print(f'  baseline (Perovskite_E2a_sad): 5.694%')
print(f'  => {"IMPROVED" if m < 0.05694 else "no improvement"} vs baseline '
      f'({(0.05694-m)*100:+.3f} pp)')


## 11. Ceiling probe — is <1% even reachable from SAD+charge on 135 samples?

The spatial map showed error is in the **valence/mid-density** region, not near-core, so
`core_weight`/`norm_rho` are counter-indicated. Before any retraining, bound the BEST POSSIBLE
ε_MAE for *any* model that learns SAD→DFT from this training set.

**Oracle k-NN probe (training-free):** for each TEST structure, retrieve the most similar
TRAINING structure(s) in SAD-feature space and use their DFT as the prediction. Grids differ in
size, so similarity uses scale-invariant SAD summary features (normalized density histogram +
total electrons + moments), matched within the same charge label. If even this oracle can't reach
<1%, the SAD+charge conditioning does not determine DFT to 1% from 135 samples — no loss tweak or
longer run fixes that. This is a lower bound on achievable error, not an estimate of the model.

In [ ]:
# CEILING PROBE — oracle k-NN in SAD-feature space (training-free lower bound on eps_MAE).
# For each TEST structure, predict its DFT from the nearest TRAINING structure(s) by SAD features,
# restricted to the same charge label. Reports the eps_MAE such an oracle would achieve.
# If >> 1%, the SAD+charge -> DFT map is not resolvable to 1% from 135 samples.
import torch, numpy as np, statistics

# scale-invariant SAD descriptor: normalized log-density histogram + total electrons + moments.
def sad_features(sad):
    s = sad.flatten().float()
    nelec = s.sum().item()
    sn = s / (s.mean() + 1e-8)                       # scale out overall magnitude
    hist = torch.histc(sn.clamp(0, 8), bins=32, min=0, max=8)
    hist = (hist / (hist.sum() + 1e-8)).cpu().numpy()
    moments = np.array([sn.mean().item(), sn.std().item(),
                        (sn**3).mean().item(), (sn**4).mean().item()])
    # log(nelec) ties structures of similar electron count together
    return np.concatenate([hist, moments, [np.log(nelec + 1e-8)]])

def load_split(prefix):
    ds = RhoDatasetCharge(
        data_list_path     = f'{LISTS_DIR}/list_d_{prefix}',
        label_list_path    = f'{LISTS_DIR}/list_l_{prefix}',
        data_gridsize_path = f'{LISTS_DIR}/list_dgs_{prefix}',
        label_gridsize_path= f'{LISTS_DIR}/list_lgs_{prefix}',
        data_augmentation  = False,
    )
    return torch.utils.data.DataLoader(ds, batch_size=1, shuffle=False)

# index the training set: features, charge, DFT-target tensor (kept on CPU)
tr_feat, tr_charge, tr_dft = [], [], []
for sad, target, charge in load_split('perovskite_train'):
    tr_feat.append(sad_features(sad[0]))
    tr_charge.append(int(charge.item()) if charge.numel()==1 else int(charge.view(-1)[0]))
    tr_dft.append(target[0].float())
tr_feat = np.stack(tr_feat)
print(f'indexed {len(tr_dft)} training structures')

K = 1   # oracle 1-NN; set 3 to test k-NN averaging (needs same-shape neighbours, see note)
eps_knn, exact_pairs = [], 0
for sad, target, charge in load_split('perovskite_test'):
    q = sad_features(sad[0]); qc = int(charge.item()) if charge.numel()==1 else int(charge.view(-1)[0])
    tgt = target[0].float()
    # candidates: same charge label AND same grid shape (so DFT tensors are comparable voxel-wise)
    cand = [j for j in range(len(tr_dft))
            if tr_charge[j]==qc and tr_dft[j].shape==tgt.shape]
    if not cand:
        continue
    d = np.linalg.norm(tr_feat[cand] - q, axis=1)
    j = cand[int(np.argmin(d))]
    pred = tr_dft[j]
    eps_knn.append(((pred - tgt).abs().sum() / tgt.sum()).item())
    if d.min() < 1e-6:
        exact_pairs += 1

m = statistics.mean(eps_knn)
print(f'
Oracle 1-NN (same charge+grid) over {len(eps_knn)} test structures:')
print(f'  mean eps_MAE: {m*100:.2f}%   median: {statistics.median(eps_knn)*100:.2f}%   '
      f'best: {min(eps_knn)*100:.2f}%')
print(f'  (exact SAD matches found for {exact_pairs} test structures)')
print()
if m < 0.01:
    print('=> Ceiling is BELOW 1%: a good model COULD reach the gate. Worth training harder.')
else:
    print(f'=> Oracle k-NN floor is {m*100:.1f}% >> 1%. Nearest-training-DFT alone cannot hit the')
    print('   gate; a learned model interpolating this set is unlikely to beat this by much.')
    print('   => <1% on 135 samples with SAD+charge conditioning is likely NOT reachable.')
